# 第7段階：検証seedによる候補選択

第6段階で得た54候補を未使用seed `40001`--`40003`で再評価し、無介入および春学期代表条件に対する頑健な改善、探索値から検証値への変化、候補領域を確認する。ここで得る値は候補選択用であり、最終性能としては報告しない。

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
STAGE_ROOT = REPO_ROOT / 'experiments/summer_2026/stage7_candidate_validation'
analysis_roots = sorted(STAGE_ROOT.glob('*/candidate_validation_analysis_v01'))
if not analysis_roots:
    raise FileNotFoundError('第7段階の正式分析がまだありません')
ANALYSIS_ROOT = analysis_roots[-1]
TABLES = ANALYSIS_ROOT / 'tables'
FIGURES = ANALYSIS_ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)

summary = json.loads((ANALYSIS_ROOT / 'analysis_summary.json').read_text())
decision = json.loads((ANALYSIS_ROOT / 'decision.json').read_text())
audit = pd.read_csv(TABLES / 'data_audit.csv')
blocks = pd.read_csv(TABLES / 'candidate_block_performance.csv')
comparison = pd.read_csv(TABLES / 'exploration_validation_comparison.csv')
effects_none = pd.read_csv(TABLES / 'candidate_effects_vs_none.csv')
ranking = pd.read_csv(TABLES / 'candidate_ranking.csv')
selected = pd.read_csv(TABLES / 'selected_candidates.csv')

NETWORK_ORDER = ['ba1000', 'facebook', 'wiki_vote']
NETWORK_LABELS = {'ba1000': 'BA1000', 'facebook': 'Facebook', 'wiki_vote': 'Wiki-vote'}
METHOD_COLORS = {'bo_gp': '#2f6b9a', 'cma_es': '#d9793f', 'random_search': '#4f8b61'}
print('analysis:', ANALYSIS_ROOT)

## 1. データ品質と選択状態

全189 runの監査状態、適格候補数、最終テストへ送る候補数を確認する。

In [ ]:
quality = pd.DataFrame({
    '項目': ['run', '有効run', 'iteration metric', '候補ブロック', '適格候補', '選択候補'],
    '件数': [summary['run_count'], summary['valid_run_count'], summary['iteration_metric_count'],
             summary['candidate_block_count'], summary['qualified_candidate_count'],
             summary['selected_candidate_count']],
})
display(quality)
display(pd.DataFrame(decision['network_decisions']))

## 2. seedブロック別の候補順位

各ネットワーク内で、行が検証seed、列が候補、色が $J_{\mathrm{cum}}$ の順位を表す。明るいほど順位が高い。候補は3ブロックの順位中央値、最悪順位、平均目的値の順に並べる。

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 7.5), constrained_layout=True)
for ax, network in zip(axes, NETWORK_ORDER):
    order = (ranking[ranking['network'] == network]
             .sort_values('overall_order')['condition_id'].tolist())
    matrix = (blocks[blocks['network'] == network]
              .pivot(index='simulator_seed', columns='condition_id', values='validation_block_rank')
              .reindex(columns=order))
    image = ax.imshow(matrix.to_numpy(), aspect='auto', cmap='viridis_r', vmin=1, vmax=18)
    ax.set_title(NETWORK_LABELS[network])
    ax.set_yticks(range(len(matrix.index)), matrix.index.astype(str))
    ax.set_ylabel('validation seed')
    ax.set_xticks(range(len(order)), order, rotation=70, ha='right', fontsize=7)
fig.colorbar(image, ax=axes, label='block rank（1が最良）', shrink=0.8)
path = FIGURES / '01_seedブロック別候補順位.png'
fig.savefig(path, dpi=180, bbox_inches='tight')
plt.show()

## 3. 探索時目的値と検証時目的値

横軸は探索seed `30001`で得た候補値、縦軸は3検証seedの平均である。破線より上なら、探索時より検証時に悪化したことを表す。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)
for ax, network in zip(axes, NETWORK_ORDER):
    frame = comparison[comparison['network'] == network]
    for method, group in frame.groupby('source_method'):
        ax.scatter(group['source_final_best'], group['validation_mean_jcum'],
                   color=METHOD_COLORS[method], label=method, s=42, alpha=0.85)
    low = min(frame['source_final_best'].min(), frame['validation_mean_jcum'].min())
    high = max(frame['source_final_best'].max(), frame['validation_mean_jcum'].max())
    ax.plot([low, high], [low, high], '--', color='#666666', linewidth=1)
    ax.set_title(NETWORK_LABELS[network])
    ax.set_xlabel('探索時 $J_{cum}$')
    ax.set_ylabel('検証時平均 $J_{cum}$')
    ax.legend(fontsize=8)
path = FIGURES / '02_探索値と検証値の比較.png'
fig.savefig(path, dpi=180, bbox_inches='tight')
plt.show()

## 4. 無介入に対する相対抑制率

点は相対抑制率 $\eta_G$、横線は階層的paired bootstrapの95% CIである。赤は第8段階へ送る代表候補、灰色は未選択候補を表す。CIは不確実性の説明に使い、候補適格性の二値閾値には使わない。

In [ ]:
effect_plot = effects_none.merge(
    ranking[['network', 'condition_id', 'overall_order', 'selected_for_final_test']],
    on=['network', 'condition_id'], validate='one_to_one')
fig, axes = plt.subplots(1, 3, figsize=(16, 7), constrained_layout=True)
for ax, network in zip(axes, NETWORK_ORDER):
    frame = effect_plot[effect_plot['network'] == network].sort_values('overall_order', ascending=False)
    y = np.arange(len(frame))
    estimate = frame['relative_suppression'].to_numpy() * 100
    low = frame['relative_ci_low'].to_numpy() * 100
    high = frame['relative_ci_high'].to_numpy() * 100
    colors = np.where(frame['selected_for_final_test'], '#b23a35', '#777777')
    for index in range(len(frame)):
        ax.errorbar(estimate[index], y[index],
                    xerr=[[estimate[index] - low[index]], [high[index] - estimate[index]]],
                    fmt='o', color=colors[index], capsize=2, markersize=4)
    ax.axvline(0, color='#333333', linewidth=1)
    ax.set_yticks(y, frame['condition_id'], fontsize=7)
    ax.set_title(NETWORK_LABELS[network])
    ax.set_xlabel('無介入に対する相対抑制率 [%]')
path = FIGURES / '03_候補別相対抑制率.png'
fig.savefig(path, dpi=180, bbox_inches='tight')
plt.show()

## 5. 候補領域と選択結果

確実性・有効性平面上に検証時平均目的値を示す。星印は第8段階へ送る代表候補である。距離0.05以内の完全連結クラスタは、観測済み候補を整理するための領域であり、領域内部の未評価点まで良いことを意味しない。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.7), constrained_layout=True)
for ax, network in zip(axes, NETWORK_ORDER):
    frame = ranking[ranking['network'] == network]
    scatter = ax.scatter(frame['certainty'], frame['effectiveness'],
                         c=frame['validation_mean_jcum'], cmap='viridis_r',
                         s=55, edgecolor='white', linewidth=0.5)
    chosen = frame[frame['selected_for_final_test']]
    ax.scatter(chosen['certainty'], chosen['effectiveness'], marker='*',
               s=220, facecolor='#d33f35', edgecolor='black', linewidth=0.8)
    for _, row in chosen.iterrows():
        ax.annotate(row['region_id'], (row['certainty'], row['effectiveness']),
                    xytext=(4, 5), textcoords='offset points', fontsize=8)
    ax.set_xlim(0.49, 1.01)
    ax.set_ylim(0.49, 1.01)
    ax.set_xlabel('確実性')
    ax.set_ylabel('有効性')
    ax.set_title(NETWORK_LABELS[network])
    fig.colorbar(scatter, ax=ax, label='検証時平均 $J_{cum}$')
path = FIGURES / '04_候補領域と選択結果.png'
fig.savefig(path, dpi=180, bbox_inches='tight')
plt.show()

display(selected[[
    'network', 'condition_id', 'region_id', 'selection_mode', 'certainty', 'effectiveness',
    'validation_mean_jcum', 'none_relative_suppression',
    'legacy_balance_absolute_suppression', 'prior_high_absolute_suppression',
]].sort_values(['network', 'selection_order']))

## 6. 解釈上の制約

- 検証seedは候補選択に使用したため、ここでの効果量を最終性能として報告しない。
- `prior_high`は先行研究CSV全体のベンチマークであり、確実性・有効性だけを変える候補の適格条件には含めない。
- 選択後は候補を追加・変更せず、未使用seed `50001`--`50005`で最終評価する。